# 01 Cleaning
Objective: load the raw inspections file, clean it, and save a processed dataset plus a one-row-per-violation table.

In [29]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/food_inspections.csv"
df = pd.read_csv(raw_path)
df.shape

(317225, 17)

## Load the data
I loaded the raw CSV and checked its shape. The dataset has 317,225 rows and 17 columns, and each row is one food inspection in Chicago.

# 02 Inspect the columns
Look at columns, types, missing values, and duplicates.

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 317225 entries, 0 to 317224
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    317225 non-null  int64  
 1   DBA Name         317225 non-null  object 
 2   AKA Name         314801 non-null  object 
 3   License #        317206 non-null  float64
 4   Facility Type    311868 non-null  object 
 5   Risk             317138 non-null  object 
 6   Address          317225 non-null  object 
 7   City             317043 non-null  object 
 8   State            317149 non-null  object 
 9   Zip              317183 non-null  float64
 10  Inspection Date  317225 non-null  object 
 11  Inspection Type  317224 non-null  object 
 12  Results          317225 non-null  object 
 13  Violations       227719 non-null  object 
 14  Latitude         316163 non-null  float64
 15  Longitude        316163 non-null  float64
 16  Location         316163 non-null  obje

# Inspect the columns
Most of the 17 columns are text. Three things need fixing:
- Inspection Date is stored as text, so I will convert it to a date type.
- License # and Zip are stored as decimal numbers (float). They are identifiers, not quantities, so I will convert them to text.
- Column names contain spaces, capital letters and a "#" symbol, so I will rename them to lowercase with underscores.
The non-null counts match my missing-values check. Violations is missing in 89,506 rows (317,225 minus 227,719).

# Clean the column names

In [31]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("#","")

#cleaned white spaces before and after the string, converted to lower, replaced spaces with _ and replaced #.

In [32]:
df.head()

,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2645405,BURRITO BEACH,BURRITO BEACH,2966529.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
1,2645427,"GRAHAM CRACKER'S DAYCARE CENTER II, INC","GRAHAM CRACKER'S DAYCARE CENTER II, INC.",2632026.0,Children's Services Facility,Risk 1 (High),1 E 113TH ST,CHICAGO,IL,60628.0,10/19/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.688848,-87.622894,"(41.688848119400404, -87.6228936894908)"
2,2645374,SIP & SAVOR COFFEE HOUSE,SIP & SAVOR,3002330.0,Restaurant,Risk 1 (High),528 E 43RD ST,CHICAGO,IL,60653.0,10/19/2026,Canvass Re-Inspection,Pass w/ Conditions,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",41.816815,-87.612923,"(41.816815018289546, -87.61292308115752)"
3,2645430,BENTO,BENTO SUSHI,3020653.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,NaN,41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
4,2645408,"BELLA EATERIES, L.L.C.",CHICK-FIL-A,3056989.0,Restaurant,Risk 1 (High),1512 E 53RD ST,CHICAGO,IL,60615.0,10/19/2026,Canvass,Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",41.799658,-87.588578,"(41.79965791698844, -87.58857814228286)"


In [33]:
df.isna().sum().sort_values(ascending=False)

violations         89506
facility_type       5357
aka_name            2424
location            1062
longitude           1062
latitude            1062
city                 182
risk                  87
state                 76
zip                   42
license_              19
inspection_type        1
inspection_date        0
results                0
dba_name               0
address                0
inspection_id          0
dtype: int64

## rename license_ to license_number

In [34]:
df = df.rename(columns={"license_":"license_number"})

In [35]:
df.head(2)

,inspection_id,dba_name,aka_name,license_number,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2645405,BURRITO BEACH,BURRITO BEACH,2966529.0,Restaurant,Risk 1 (High),5700 S MARYLAND AVE,CHICAGO,IL,60637.0,10/19/2026,Canvass,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.791261,-87.605085,"(41.79126061783551, -87.60508491200589)"
1,2645427,"GRAHAM CRACKER'S DAYCARE CENTER II, INC","GRAHAM CRACKER'S DAYCARE CENTER II, INC.",2632026.0,Children's Services Facility,Risk 1 (High),1 E 113TH ST,CHICAGO,IL,60628.0,10/19/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.688848,-87.622894,"(41.688848119400404, -87.6228936894908)"


## Check duplicates

In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:
##There are no duplicated in the dataset

## Checking duplicate inspection ids

In [38]:
df["inspection_id"].duplicated().sum()

np.int64(0)

No duplicate inspection ids found.

## Dates and text
Convert the date, tidy text columns, and look at the result values.

In [39]:
df["dba_name"] = df["dba_name"].astype("string").str.strip()
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
for col in ["facility_type", "risk", "results"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()
df["results"].value_counts()

results
Pass                    164230
Fail                     61090
Pass W/ Conditions       47011
Out Of Business          25944
No Entry                 14217
Not Ready                 4637
Business Not Located        96
Name: count, dtype: Int64

## Convert zip and license to text

In [40]:
df.columns.tolist()

['inspection_id',
 'dba_name',
 'aka_name',
 'license_number',
 'facility_type',
 'risk',
 'address',
 'city',
 'state',
 'zip',
 'inspection_date',
 'inspection_type',
 'results',
 'violations',
 'latitude',
 'longitude',
 'location']

In [41]:
df["zip"] = df["zip"].astype("Int64").astype("string")
print(df["zip"].head())
df["license_number"] = df["license_number"].astype("Int64").astype("string")
print(df["license_number"].head())

0    60637
1    60628
2    60653
3    60637
4    60615
Name: zip, dtype: string
0    2966529
1    2632026
2    3002330
3    3020653
4    3056989
Name: license_number, dtype: string


Zip and license number were decimals because of missing values. They are identifiers, not quantities, so I converted both to text.

## Fix data types and standardize text
Converted inspection_date to a datetime and standardized the capitalization of facility_type, risk and results so the same value is not counted twice. I kept dba_name's original capitalization because business names like "McDonald's" would be damaged. The results column now has 7 clean categories, led by Pass (164,230) and Fail (61,090).

In [42]:
df[df["violations"].isna()]["results"].value_counts()

results
Pass                    40913
Out Of Business         25900
No Entry                13437
Not Ready                4554
Fail                     3655
Pass W/ Conditions        951
Business Not Located       96
Name: count, dtype: Int64

#### Missing violations
I checked the results for rows with no violations recorded. About 49% were inspections that did not happen (Out Of Business, No Entry, Not Ready, 
Business Not Located) and about 46% were Pass, so an empty Violations field mostly means nothing was found. A small group of 4,606 rows (about 5%) are 
Fail or Pass W/ Conditions with no violations listed, which may be a gap in the source data. I kept all of these rows and will handle the violations 
text when I split it.

## Create analysis columns


In [43]:
df['results'].value_counts()

results
Pass                    164230
Fail                     61090
Pass W/ Conditions       47011
Out Of Business          25944
No Entry                 14217
Not Ready                 4637
Business Not Located        96
Name: count, dtype: Int64

## Check the results column
The results column has 7 clean categories and no missing values. Pass is the most common (52%), followed by Fail (19%) and Pass W/ Conditions (15%). The other four categories are cases where no inspection happened, so I will leave them out of pass and fail rates.

In [44]:
df["is_fail"] = df["results"].eq("Fail")
df["inspection_year"] = df["inspection_date"].dt.year
df["inspection_month"] = df["inspection_date"].dt.month
df[["results", "is_fail", "inspection_year", "inspection_month"]].head(10)

,results,is_fail,inspection_year,inspection_month
0,Pass,False,2026,10
1,Pass,False,2026,10
2,Pass W/ Conditions,False,2026,10
3,Pass,False,2026,10
4,Pass,False,2026,10
5,Pass,False,2026,10
6,Pass,False,2026,10
7,Pass,False,2026,10
8,Pass W/ Conditions,False,2026,10
9,Pass,False,2026,10


## Create new columns
I added three columns to make analysis easier. is_fail is True when the result is Fail and False for everything else. inspection_year and inspection_month are taken from the inspection date, so I can look at trends over time. The first 10 rows are all from October 2026 and none failed, so I will check the full column before drawing conclusions.

In [45]:
df['is_fail'].value_counts(normalize=True)

is_fail
False    0.807424
True     0.192576
Name: proportion, dtype: Float64

## Check the failure rate
About 19.3% of inspections failed (0.1926) and 80.7% did not. This matches the 19% Fail share I found in the results column, so the is_fail column was created correctly.

In [46]:
#df["is_fail"].mean()

In [47]:
real = df[df["results"].isin(["Pass", "Pass W/ Conditions", "Fail"])]
real["is_fail"].mean()

np.float64(0.22432260741524102)

Since this includes inspections that did not take place, the failure rate among real inspections will be higher, and I will check that in the EDA.

In [48]:
## Check is_fail by result

In [49]:
df.groupby('results')['is_fail'].sum()

results
Business Not Located        0
Fail                    61090
No Entry                    0
Not Ready                   0
Out Of Business             0
Pass                        0
Pass W/ Conditions          0
Name: is_fail, dtype: Int64

To confirm the is_fail flag works, I counted it by inspection result. It is True for all 61,090 Fail rows and zero for every other result, so the flag matches the source data. I treated Pass W/ Conditions as a pass, because the business was allowed to operate. This definition will drive the failure-rate analysis later.

## Split the violations text
Each inspection has many violations in one cell, separated by `|`. Make one row per violation.

## Create the violations table

In [50]:
viol = df[["inspection_id", "violations"]].dropna().copy()
viol.shape

(227719, 2)

I made a separate table with only inspection_id and violations, keeping the 227,719 inspections that have violations text. The 89,506 inspections with no violations are left out of this table but remain in the main dataset. Next I will split the text so each violation gets its own row.

In [51]:
viol["violations"] = viol["violations"].str.split("|", regex=False)
viol = viol.explode("violations")
viol["violations"] = viol["violations"].str.strip()
viol = viol[viol["violations"] != ""]

# Pull out the violation number and description; drop the inspector's comments
viol["violation_code"] = viol["violations"].str.extract(r"^(\d+)\.")[0]
viol["violation_description"] = viol["violations"].str.extract(r"^\d+\.\s*(.*?)(?:\s*-\s*Comments:|$)")[0]
viol = viol.drop(columns="violations")
viol.head(10)



,inspection_id,violation_code,violation_description
0,2645405,38,"INSECTS, RODENTS, & ANIMALS NOT PRESENT"
0,2645405,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN"
1,2645427,10,ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED A...
1,2645427,38,"INSECTS, RODENTS, & ANIMALS NOT PRESENT"
1,2645427,38,"INSECTS, RODENTS, & ANIMALS NOT PRESENT"
1,2645427,47,"FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PR..."
1,2645427,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN"
1,2645427,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN"
1,2645427,56,ADEQUATE VENTILATION & LIGHTING; DESIGNATED AR...
1,2645427,57,ALL FOOD EMPLOYEES HAVE FOOD HANDLER TRAINING


## Extract violation code and description
I used regular expressions to pull the violation number and the description out of each violation's text, and left out the inspector's comments. I then dropped the original combined column. The table now has inspection_id, the full violation text, violation_code and violation_description.

In [52]:
print(viol.shape)
print(viol["inspection_id"].nunique())
print(viol[["violation_code", "violation_description"]].isna().sum())

(1027296, 3)
227719
violation_code           0
violation_description    0
dtype: int64


## Extract violation code and description
I pulled the violation number and description out of each violation's text using regular expressions and dropped the inspector's comments. The split produced 1,027,296 violation rows from 227,719 inspections, which is about 4.5 violations per inspection. Every row has both a code and a description, so the patterns matched everything.

In [53]:
viol.to_csv("../data/processed/violations.csv", index=False)

## Check and save
Check row counts and a few random rows before saving.

In [54]:
# Choose the columns to keep
columns_to_keep = [
    "inspection_id", "dba_name", "license_number", "facility_type",
    "risk", "zip", "inspection_date", "inspection_type",
    "results", "is_fail", "latitude", "longitude"
]
# final inspections table
inspections = df[columns_to_keep]
print(inspections.shape)
inspections.sample(5, random_state=1)



(317225, 12)


,inspection_id,dba_name,license_number,facility_type,risk,zip,inspection_date,inspection_type,results,is_fail,latitude,longitude
122793,2312731,CATCH THIRTY FIVE,19903,Restaurant,Risk 1 (High),60601,2019-09-23,Canvass,Pass W/ Conditions,False,41.886710,-87.629303
165461,2059970,THE PORTAGE,2022387,Restaurant,Risk 1 (High),60634,2017-06-09,Canvass,No Entry,False,41.952525,-87.767127
283090,659230,"SAMAH DISCOUNT STORE, INC",2125353,Grocery Store,Risk 3 (Low),60609,2011-11-04,License,Fail,True,41.810070,-87.665159
181,2644145,LARDON & THE UNION TAPROOM & THE MEADOWLARK,2712521,Restaurant,Risk 1 (High),60647,2026-10-13,Complaint Re-Inspection,Pass,False,41.921724,-87.697468
85885,2546708,SUSHI-SAN,2812547,Restaurant,Risk 1 (High),60654,2022-01-20,License,Pass,False,41.891525,-87.630519


In [55]:
# look at the violations table too
print(viol.shape)
viol.sample(5, random_state=1)

(1027296, 3)


,inspection_id,violation_code,violation_description
197392,1609810,41,"PREMISES MAINTAINED FREE OF LITTER, UNNECESSAR..."
285709,634614,32,FOOD AND NON-FOOD CONTACT SURFACES PROPERLY DE...
225452,1441452,12,HAND WASHING FACILITIES: WITH SOAP AND SANITAR...
279073,670215,37,TOILET ROOM DOORS SELF CLOSING: DRESSING ROOMS...
141725,2213772,5,PROCEDURES FOR RESPONDING TO VOMITING AND DIAR...


In [56]:
#save both tables
inspections.to_csv("../data/processed/inspections_clean.csv", index=False)
viol.to_csv("../data/processed/violations_clean.csv", index=False)

I kept the 12 columns I need and saved two files: inspections_clean.csv (one row per inspection) and violations_clean.csv (one row per violation). They link through inspection_id. I checked the row counts and random rows before saving so the files are ready for SQL and Tableau.